## LLM Judge Evaluation with Microsoft Foundry

### Installing Utilities and Libraries

In [ ]:
%pip install azure-ai-projects==2.7.0 openai==1.109.1 python-dotenv azure-identity

### Setting up the Environment Variables

In [ ]:
import json
import time
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import TestingCriterionAzureAIEvaluator
from openai.types.eval_create_params import DataSourceConfigCustom
from openai.types.evals.create_eval_jsonl_run_data_source_param import CreateEvalJSONLRunDataSourceParam, SourceFileID
from azure.ai.projects.models import EvaluatorCategory, EvaluatorDefinitionType
import os
from dotenv import load_dotenv

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("CHAT_MODEL_DEPLOYMENT_NAME")

### Create the Foundry Project Client

In [ ]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential()
)

### Create the OpenAI Client

In [ ]:
openai_client = project_client.get_openai_client()

### Upload the Eval Dataset to Microsoft Foundry 

In [ ]:
dataset_name = "ticket-classification-eval-dataset"
dataset_version = "1"

dataset = project_client.datasets.upload_file(
    name = dataset_name, 
    version = dataset_version,
    file_path = "./ticket_classification_eval.jsonl"
)

data_id = dataset.id

print("Dataset ID:", data_id)

### Define the Evaluation Dataset Schema

In [ ]:
data_source_config = DataSourceConfigCustom(

    type="custom",

    item_schema={

        "type": "object",

        "properties": {

            "ticket": {
                "type": "string"
            },

            "output": {
                "type": "string"
            },

            "expected_category": {
                "type": "string"
            },

            "expected_priority": {
                "type": "string"
            },
        },

        "required": [
            "ticket",
            "output",
            "expected_category",
            "expected_priority",
        ],
    },
)

### Create the Classification Accuracy LLM Judge

In [ ]:
classification_accuracy_instructions = """
Evaluate whether the generated IT support ticket
classification matches the expected classification.

Generated classification:

{{output}}

Expected category:

{{expected_category}}

Expected priority:

{{expected_priority}}

Return true ONLY if BOTH:

1. The generated category matches expected_category.

2. The generated priority matches expected_priority.

Otherwise return false.
"""

In [ ]:
classification_accuracy_evaluator = project_client.beta.evaluators.create_version(

        name="classification_accuracy",

        evaluator_version={

            "name":
                "classification_accuracy",

            "display_name":
                "Classification Accuracy",

            "description":
                (
                    "Evaluates whether the generated "
                    "category and priority match the "
                    "expected values."
                ),

            "categories": [
                EvaluatorCategory.QUALITY
            ],

            "definition": {

                "type":
                    EvaluatorDefinitionType.PROMPT,

                "prompt_text":
                    classification_accuracy_instructions,

                "init_parameters": {

                    "type": "object",

                    "properties": {

                        "deployment_name": {
                            "type": "string"
                        }
                    },

                    "required": [
                        "deployment_name"
                    ]
                },

                "data_schema": {

                    "type": "object",

                    "properties": {

                        "output": {
                            "type": "string"
                        },

                        "expected_category": {
                            "type": "string"
                        },

                        "expected_priority": {
                            "type": "string"
                        }
                    },

                    "required": [
                        "output",
                        "expected_category",
                        "expected_priority"
                    ]
                },

                "metrics": {

                    "classification_accuracy": {

                        "type":
                            "boolean",

                        "desirable_direction":
                            "increase"
                    }
                }
            }
        }
    )

### Create the Output Format Compliance LLM Judge

In [ ]:
output_format_instructions = """
Evaluate whether the generated output follows exactly
this structure:

Category: <category>
Priority: <priority>

The category must be one of:

Network
Authentication
Hardware
Software
Security

The priority must be one of:

Low
Medium
High
Critical

Generated output:

{{output}}

There must not be any additional explanation or text.

Return true if the output follows all requirements.

Otherwise return false.
"""

In [ ]:
output_format_evaluator = project_client.beta.evaluators.create_version(

        name="output_format_compliance",

        evaluator_version={

            "name":
                "output_format_compliance",

            "display_name":
                "Output Format Compliance",

            "description":
                (
                    "Checks whether ticket classifications "
                    "follow the required output structure."
                ),

            "categories": [
                EvaluatorCategory.QUALITY
            ],

            "definition": {

                "type":
                    EvaluatorDefinitionType.PROMPT,

                "prompt_text":
                    output_format_instructions,

                "init_parameters": {

                    "type": "object",

                    "properties": {

                        "deployment_name": {
                            "type": "string"
                        }
                    },

                    "required": [
                        "deployment_name"
                    ]
                },

                "data_schema": {

                    "type": "object",

                    "properties": {

                        "output": {
                            "type": "string"
                        }
                    },

                    "required": [
                        "output"
                    ]
                },

                "metrics": {

                    "output_format_compliance": {

                        "type":
                            "boolean",

                        "desirable_direction":
                            "increase"
                    }
                }
            }
        }
    )

### Configure the Evaluators for the Cloud Evaluation

In [ ]:
testing_criteria = [

    TestingCriterionAzureAIEvaluator(

        type="azure_ai_evaluator",

        name="classification_accuracy",

        evaluator_name="classification_accuracy",

        initialization_parameters={
            "deployment_name": model_deployment_name
        },

        data_mapping={

            "output":"{{item.output}}",

            "expected_category":"{{item.expected_category}}",

            "expected_priority":"{{item.expected_priority}}",
        },
    ),


    TestingCriterionAzureAIEvaluator(

        type="azure_ai_evaluator",

        name="output_format_compliance",

        evaluator_name="output_format_compliance",

        initialization_parameters={
            "deployment_name": model_deployment_name
        },

        data_mapping={

            "output": "{{item.output}}",
        },
    ),
]

### Create the Cloud Evaluation

In [ ]:
eval_object = openai_client.evals.create(
    name = "IT Ticket Classification Evaluation",
    data_source_config = data_source_config,
    testing_criteria = testing_criteria
)

print("Evaluation ID:", eval_object.id)

### Run the Evaluation

In [ ]:
eval_run = openai_client.evals.runs.create(

        eval_id = eval_object.id,

        name = "dataset-run",

        data_source = CreateEvalJSONLRunDataSourceParam(
            type = "jsonl",
            source = SourceFileID(
                        type = "file_id",
                        id = data_id
                )
        )
)

print("Evaluation run has been triggered!")